# GEM Pipeline & Terminal Data Export

Exports LNG terminals and pipelines from Google Sheets to Excel/GeoJSON/GeoPackage.

**Pipeline modes:** Oil-NGL | Oil | NGL | Gas | Gas-Hydrogen | Hydrogen | Oil-and-Gas (combined)

**Optional:** Filter by country list

All logic lives in `pipeline_exports.py` (same directory) — this notebook is a thin interactive wrapper around it. The same module powers the CI map build (`.github/workflows/build-map-data.yml`) and can be run directly: `python pipeline_exports.py --help`.

In [ ]:
%pip install -q -e ../../gem-tracker-constants
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# All export logic lives in pipeline_exports.py (same directory), shared
# with the CI map build. autoreload picks up module edits without a
# kernel restart.
from pipeline_exports import (
    refresh_routes,
    build_release,
    report_id_reconciliation,
    release_base_name,
    export_files,
    write_map_geojson,
)

print("✓ Imports loaded")

## Configuration

In [ ]:
# Sheets
PIPELINES_SHEET_KEY = '1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek'
#PIPELINES_SHEET_KEY = '1gChRPYLrcirx3lNI_DHWs5GaKTtfXgELVg1bHix8zqs' # june 2026 goit update

# Pipeline type: 'Oil-NGL' | 'Oil' | 'NGL' | 'Gas' | 'Gas-Hydrogen' | 'Hydrogen' | 'Oil-and-Gas'
#PIPELINE_TYPE = 'Oil-NGL'
#PIPELINE_TYPE = 'Oil'
#PIPELINE_TYPE = 'NGL'
#PIPELINE_TYPE = 'Oil-and-Gas'
PIPELINE_TYPE = 'Gas'

# Fuel simplification: None | 'Oil' | 'NGL' | 'Oil-and-NGL' | 'Gas'
#SIMPLIFY_FUELS = 'Oil-and-NGL'
#SIMPLIFY_FUELS = None
#SIMPLIFY_FUELS = 'Oil'
#SIMPLIFY_FUELS = 'NGL'
SIMPLIFY_FUELS = 'Gas'

# Status filter (None = all statuses, or list of status values)
FILTER_STATUS = None
#FILTER_STATUS = ['operating']
#FILTER_STATUS = ['operating', 'construction']

# Geographic filter (None = all countries, or list of country names)
FILTER_COUNTRIES = None
#FILTER_COUNTRIES = ['Iran', 'Iraq', 'Saudi Arabia', 'Kuwait', 'United Arab Emirates', 'Qatar', 'Oman']

# Paths
# Routes come from a git worktree pinned to the 'normalized' branch of
# goit-ggit-pipeline-routes (the main checkout stays on whatever branch
# it's on). refresh_routes() auto-pulls it each run so exports use the
# latest published geometries.
PIPELINE_ROUTES_PATH = (
    '/Users/baird/Dropbox/_git_ALL/_github-repos-gem/'
    'goit-ggit-pipeline-routes-normalized/data/individual-routes/'
)
OUTPUT_DIR = 'data-files'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

refresh_routes(PIPELINE_ROUTES_PATH)

print(f"✓ Config: {PIPELINE_TYPE}" + (f" | Status: {FILTER_STATUS}" if FILTER_STATUS else "") + (f" | Filter: {len(FILTER_COUNTRIES)} countries" if FILTER_COUNTRIES else ""))

## Helper functions

Moved to `pipeline_exports.py` (imported above): `get_config`, `fetch_pipeline_data`, `filter_by_countries`, `filter_by_status`, `simplify_fuel_types`, `check_no_route_geojson_files`, `enforce_no_route_null_geometry`, `load_geometries`, `export_files`, plus the orchestration (`build_release`, `report_id_reconciliation`) and map output (`write_map_geojson`).

---
## Part 1: Terminals

In [12]:
# # Load from Google Sheets
# gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
# ss = gc.open_by_key(TERMINALS_SHEET_KEY)

# terms_df = ss.worksheet('title', 'LNG export & import terminals').get_as_df(start='A3')
# terms_dict = ss.worksheet('title', 'Data dictionary - Terminals').get_as_df()
# terms_acro = ss.worksheet('title', 'Acronyms').get_as_df()
# terms_copy = ss.worksheet('title', 'Copyright - GGIT').get_as_df()
# if terms_copy.shape[1] > 1:
#     terms_copy = pd.DataFrame(terms_copy.iloc[:, 0])

# # Filter
# orig_count = len(terms_df)
# terms_df = terms_df[
#     (terms_df['TerminalType'] != 'Oil export terminal') &
#     (terms_df['TerminalName'] != '') &
#     (terms_df['Status'] != 'N/A')
# ]

# # Select columns for export
# cols_to_export = terms_dict[
#     (terms_dict['IncludeWithDataRelease'] == 'Yes') &
#     (terms_dict['DataReleaseColumnOrder'].notna())
# ].sort_values('DataReleaseColumnOrder')['VariableName'].tolist()

# terms_dict_export = terms_dict[
#     terms_dict['VariableName'].isin(cols_to_export)
# ][['VariableName', 'Definition']]

# # Create geometries
# terms_df['geometry'] = terms_df.apply(
#     lambda r: shapely.geometry.Point(r['Longitude'], r['Latitude']) 
#     if r['Longitude'] not in ['Unknown', 'TBD', ''] and r['Latitude'] not in ['Unknown', 'TBD', ''] 
#     else shapely.geometry.Point(),
#     axis=1
# )

# terms_gdf = gpd.GeoDataFrame(terms_df[cols_to_export], geometry=terms_df['geometry'], crs='EPSG:4326')

# print(f"✓ Terminals: {orig_count} → {len(terms_gdf)} (after filtering)")

# # Export (date stamp: year-month only)
# today = datetime.date.today()
# term_files = export_files(
#     terms_gdf,
#     f"{OUTPUT_DIR}/GEM-GGIT-LNG-Terminals-{today:%Y-%m}",
#     terms_dict_export, terms_acro, terms_copy
# )

---
## Part 2: Pipelines

In [ ]:
release = build_release(
    pipeline_type=PIPELINE_TYPE,
    routes_path=PIPELINE_ROUTES_PATH,
    simplify_fuels=SIMPLIFY_FUELS,
    filter_status=FILTER_STATUS,
    filter_countries=FILTER_COUNTRIES,
    sheet_key=PIPELINES_SHEET_KEY,
)

config = release['config']
pipes_gdf = release['gdf_full']
pipes_gdf_export = release['gdf_export']
pipes_dict_export = release['dict_export']
pipes_acro = release['acronyms']
pipes_copy = release['copyright']
raw_db_ids = release['raw_db_ids']

### Compare database ProjectIDs with GitHub repo routes

In [ ]:
report_id_reconciliation(pipes_gdf, raw_db_ids, PIPELINE_ROUTES_PATH, PIPELINE_TYPE)

In [ ]:
# Export pipelines
today = datetime.date.today()
base_name = release_base_name(OUTPUT_DIR, config, PIPELINE_TYPE,
                              FILTER_STATUS, FILTER_COUNTRIES, date=today)

pipe_files = export_files(
    pipes_gdf_export,
    base_name,
    pipes_dict_export, pipes_acro, pipes_copy
)

---
## Summary

In [ ]:
print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print(f"Date: {today}")
#print(f"\nTerminals: {len(terms_gdf)}")
print(f"Pipelines: {len(pipes_gdf_export)} ({PIPELINE_TYPE})" + (f" | Simplified: {SIMPLIFY_FUELS}" if SIMPLIFY_FUELS else ""))
if FILTER_STATUS:
    print(f"Status: {', '.join(FILTER_STATUS)}")
if FILTER_COUNTRIES:
    print(f"Countries: {', '.join(FILTER_COUNTRIES)}")
#print(f"\nFiles: {len(term_files) + len(pipe_files)}")
print(f"\nFiles: {len(pipe_files)}")
print("=" * 60)

EXPORT COMPLETE
Date: 2026-06-15
Pipelines: 1926 (Oil-NGL) | Simplified: Oil-and-NGL

Files: 4
